# 00 — Setup checks

Run this first, on any machine, before any experiment. It confirms the four
things every downstream result silently depends on:

1. `swiftbench` imports and the frozen split loads.
2. The five language files still share the frozen schema.
3. Sentiment and priority are still identical across languages for a given `id`
   (they are labeled once in English and copied — if that drifts, cross-language
   comparisons stop meaning anything).
4. Train and dev share no `id`.

If anything here fails, stop. Do not run experiments on top of a broken invariant.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import pandas as pd
import matplotlib.pyplot as plt

import swiftbench as sb

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Stamped into every result file so three people's runs stay attributable.
AUTHOR = ""

manifest = sb.splits.ensure()
print("split sha:", manifest["sha"], manifest["counts"])

split sha: e7b5934392cd {'train': 8500, 'dev': 1498, 'test': 3079}


## Schema and id-alignment

`category_mismatch`, `sentiment_mismatch` and `priority_mismatch` must all be **0**.
Non-zero means one language file was edited in isolation.

In [2]:
alignment = sb.data.check_alignment()
display(alignment)

assert (alignment[["category_mismatch", "sentiment_mismatch", "priority_mismatch"]] == 0).all().all(), \
    "labels have drifted between languages — fix before running anything"
assert (alignment[["ids_only_in_english", "ids_only_in_other"]] == 0).all().all(), \
    "language files no longer cover the same ids"
print("id alignment OK across all five languages")

,portion,language,ids_in_common,ids_only_in_english,ids_only_in_other,category_mismatch,sentiment_mismatch,priority_mismatch
0,train,sinhala,9998,0,0,0,0,0
1,train,singlish,9998,0,0,0,0,0
2,train,tamil,9998,0,0,0,0,0
3,train,tamilish,9998,0,0,0,0,0
4,test,sinhala,3079,0,0,0,0,0
5,test,singlish,3079,0,0,0,0,0
6,test,tamil,3079,0,0,0,0,0
7,test,tamilish,3079,0,0,0,0,0


id alignment OK across all five languages


## Split integrity

Dev is carved out of the official BANKING77 *train* file. Test is the official
*test* file and is not touched during model selection.

In [3]:
train = sb.splits.get(sb.config.LANGUAGES, "train")
dev = sb.splits.get(sb.config.LANGUAGES, "dev")
test = sb.splits.get(sb.config.LANGUAGES, "test")

print(f"train rows {len(train):6d}   unique ids {train.id.nunique():5d}")
print(f"dev   rows {len(dev):6d}   unique ids {dev.id.nunique():5d}")
print(f"test  rows {len(test):6d}   unique ids {test.id.nunique():5d}")

assert not set(train.id) & set(dev.id), "train/dev id overlap — split is leaking"
print("\ntrain/dev share no id — the dev carve-out is clean")

# `id` is a 0-based row index into original-dataset/train.csv and test.csv
# *separately*, so train ids and test ids overlap numerically while referring to
# different tickets. Comparing them as sets is meaningless; the separation that
# matters is that they come from different source files.
print("note: ids are file-local (train.csv and test.csv each start at 0),")
print("      so train/test id ranges overlap by construction. Not a leak.")

print("\nEach id appears once per language, so row counts are 5x the id counts:")
print(train.groupby("language").size())

train rows  42500   unique ids  8500
dev   rows   7490   unique ids  1498
test  rows  15395   unique ids  3079

train/dev share no id — the dev carve-out is clean
note: ids are file-local (train.csv and test.csv each start at 0),
      so train/test id ranges overlap by construction. Not a leak.

Each id appears once per language, so row counts are 5x the id counts:
language
english     8500
singlish    8500
sinhala     8500
tamil       8500
tamilish    8500
dtype: int64


## Label distributions

The two numbers that shape how every result in this project gets reported.

In [4]:
for task in ["sentiment", "priority"]:
    col = sb.data.label_column(task)
    dist = dev[dev.language == "english"][col].value_counts()
    share = (dist / dist.sum() * 100).round(1)
    print(f"{task}:")
    for label in dist.index:
        print(f"   {label:9s} {dist[label]:5d}  ({share[label]:.1f}%)")
    print()

print("Consequence: a model that always answers 'Neutral' scores ~95.5% sentiment")
print("accuracy and catches zero angry customers. Sentiment is reported as")
print("Negative-class F1 throughout. Priority is reported as macro-F1.")

sentiment:
   Neutral    1430  (95.5%)
   Negative     68  (4.5%)

priority:
   Low         790  (52.7%)
   Medium      556  (37.1%)
   High        152  (10.1%)

Consequence: a model that always answers 'Neutral' scores ~95.5% sentiment
accuracy and catches zero angry customers. Sentiment is reported as
Negative-class F1 throughout. Priority is reported as macro-F1.
